In [ ]:
%pip install -q otter-grader

# Case Study 14: Did the Community Health Worker Program Reduce ER Visits?

**DS701 — Session 18 (Wed Nov 4, 2026) — Section A1**

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/class_activity_notebooks/14-InClass-Exercise-Causal-Inference-A1/14-InClass-Exercise-Causal-Inference-A1.ipynb)

**Time: ~60 minutes.** Work in groups of 2-3.

Parts marked **(autograded)** are submitted to Gradescope; open-ended parts are graded
for participation. You may use AI assistance, but you must be able to explain and
justify every part of your solution when asked.

**Plan**

| Part | The health department's question | Method | Grading | Time |
|---|---|---|---|---|
| 1 | Enrollment is highest in our sickest neighborhoods. Does the citywide table mislead us, and what does the adjusted one say? | Simpson's paradox by hand; direct standardization | autograded + short discussion | ~15 min |
| 2 | In our analysts' three diagrams, which routes let *who enrolled* pass for a program effect — and are they open? | Every backdoor path by hand, marked open / blocked | autograded | ~12 min |
| 3 | Which variables in our records should we adjust for, and which must we leave alone? | One valid and one *invalid* adjustment set per diagram | autograded + short discussion | ~13 min |
| — | Report out: what you would tell the commissioner | team memo, then pair-share | participation | ~10 min |

Dependencies: `numpy`, `pandas`, `matplotlib`, `networkx`.

## Case Study Setup

You are a data scientist on loan to a city public-health department. Last year the
department ran a **community health worker (CHW) program**: trained outreach workers
visited enrolled residents at home, helped them manage chronic conditions and keep
primary-care appointments, in the hope that fewer of them would end up in the emergency
room. Enrollment was voluntary, and outreach went first to the neighborhoods with the
most need. The program's analysts now report that **enrolled residents used the ER
less**; the city budget office, reading the same citywide totals, says they used it
*more*. The commissioner must decide in two weeks whether to fund a second year, and
wants one question answered: *did the program cause a drop in ER visits, or did it just
enroll different people?*

Nothing here is real department data, and nothing is downloaded. The Part 1 table is a
hand-sized **constructed** example: 700 residents, with counts chosen so the arithmetic
is clean, built to have the structure the department describes (two groups, two
neighborhood strata, one outcome). The three diagrams in Parts 2 and 3 are candidate
pictures of how the department's variables cause one another — the kind of sketch its
analysts would draw, written for this exercise rather than estimated from anything.
Because the numbers and the arrows were chosen, every step can be checked by hand.

**What the health department asked you, and where you answer it**

1. *Enrollment is highest in our sickest neighborhoods. Does the citywide table mislead
   us, and what does it say once that is taken into account?* — Part 1
2. *Our analysts could not agree on one diagram of how enrollment, ER use and the rest of
   our records relate. In each candidate, which routes let the difference between
   enrollees and everyone else pass for a program effect — and are they open?* — Part 2
3. *Which of the variables in our records should we adjust for, and which must we leave
   alone?* — Part 3
4. *What number goes to the funding meeting, and what would we have to believe for it to
   be the program's effect?* — Report out

The vocabulary is the lecture's: a **confounder** is a common cause of treatment and
outcome; the **selection-bias term** is the baseline difference
$\mathbb{E}[Y(0)\mid T{=}1] - \mathbb{E}[Y(0)\mid T{=}0]$ between those who were and
were not treated; a **backdoor path** is a path from treatment to outcome whose first
arrow points *into* the treatment; **collider bias** is what conditioning on a common
*effect* produces. Nothing today is random, so the seed in the next cells is unused.

Run the setup cells that follow as-is; everything below depends on them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
# Lecture section. Only A1 runs this semester; this is kept because the build
# script reads it to name the variant directory the notebook is published in.
SECTION = "A1"
SEED = 701
rng = np.random.default_rng(SEED)

In [ ]:
# Colab: fetch the autograder tests for this activity.
import os, sys, urllib.request

if "google.colab" in sys.modules:
    os.makedirs("tests", exist_ok=True)
    BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/class_activity_notebooks/14-InClass-Exercise-Causal-Inference-A1/tests/"
    for _q in ("q1", "q2", "q4", "q5"):
        urllib.request.urlretrieve(BASE + _q + ".py", "tests/" + _q + ".py")

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook()

## Part 1 (autograded): Simpson's paradox in the program table — and how to *adjust* it

**Why.** The department's first question: *enrollment is highest in our sickest
neighborhoods — does the citywide table mislead us, and what does it say once that is
taken into account?* The analysts and the budget office are reading the **same** table
and reaching opposite conclusions, which is the signature of Simpson's paradox. You will
recompute both readings by hand, find the confounder's two fingerprints, and then
standardize the table so the two groups are compared at the same neighborhood mix.

The analysts laid the table out the way the lecture's kidney-stone study was (two arms,
two strata), so you can check your arithmetic against it:

| in the code | in the program |
|---|---|
| `A` / `B` (the two **arms**) | **enrolled** / **not enrolled** |
| `low` / `high` (the two **strata**) | **low-need** / **high-need** neighborhood — the strata outreach was targeted by |
| `(no_er, total)` | residents with **no ER visit** in the program year, residents in that cell |

Below are your section's counts.

In [ ]:
COUNTS = {("A", "low"):  (45, 50),   ("A", "high"): (210, 300),
          ("B", "low"):  (240, 300),  ("B", "high"): (30, 50)}

ARMS = ("A", "B")
STRATA = ("low", "high")

table = pd.DataFrame(
    {a: {s: f"{COUNTS[(a, s)][0]} / {COUNTS[(a, s)][1]}" for s in STRATA} for a in ARMS}
).T
table.index.name = "arm"
print(f"Section {SECTION}: residents with no ER visit / residents")
table

**Task.** Using only `COUNTS` (arithmetic, no libraries needed):

1. `rate_within` — a dict mapping each `(arm, stratum)` key to its no-ER-visit rate
   (`no_er / total`, a float in $[0,1]$).
2. `rate_overall` — a dict mapping each arm to its **pooled** (citywide) rate
   (all residents with no ER visit ÷ all residents in that arm).
3. `better_within` — the arm (`"A"` or `"B"`) with the higher rate in **both** strata;
   `better_overall` — the arm with the higher pooled rate.
4. `reversal` — `True` if `better_within != better_overall`, else `False`.
5. `share_high` — a dict mapping each arm to the fraction of *its* residents who live in
   high-need neighborhoods.

Then look at `share_high` and say, in one sentence, why the analysts (within strata) and
the budget office (citywide) disagree.

In [ ]:
rate_within = ...
rate_overall = ...
                for a in ARMS}
better_within = ...
better_overall = ...
reversal = ...
share_high = ...

for a in ARMS:
    print(f"arm {a}: " + ", ".join(f"{s}-need {rate_within[(a, s)]:.1%}" for s in STRATA)
          + f", overall {rate_overall[a]:.1%}, share high-need {share_high[a]:.0%}")
print("better within each stratum:", better_within, "| better overall:", better_overall, "| reversal:", reversal)

In [ ]:
grader.check("q1")

### Adjusting the table (autograded)

"Hold the confounder fixed" has a numerical version for tables, called **direct
standardization**: give *both* arms the **same** neighborhood mix and recompute. Use the
citywide mix — the fraction of *all* residents in the table (both arms together) who
live in each stratum — as the common weights:

$$
\text{adjusted}(a) \;=\; \sum_{s \in \{\text{low},\,\text{high}\}}
      \text{rate\_within}(a, s)\;\cdot\;P(s)\quad\text{where } P(s)=\frac{\text{all residents in stratum } s}{\text{all residents}}.
$$

**Task.** Compute `p_stratum` (dict: stratum → citywide fraction), `rate_adjusted` (dict:
arm → standardized no-ER-visit rate) and `better_adjusted` (`"A"` or `"B"`).
This is what `Y ~ T + Z` does for you in a regression: it compares enrolled and
not-enrolled residents at a *common* value of the confounder.

In [ ]:
n_all = sum(n for (_, n) in COUNTS.values())
p_stratum = ...
rate_adjusted = ...
better_adjusted = ...


print("citywide neighborhood mix:", {s: round(p, 3) for s, p in p_stratum.items()})
for a in ARMS:
    print(f"arm {a}: pooled {rate_overall[a]:.1%}  ->  adjusted {rate_adjusted[a]:.1%}")
print("better after adjustment:", better_adjusted)

In [ ]:
grader.check("q2")

<!-- BEGIN QUESTION -->

### Discuss (participation)

The commissioner will carry **one number** into the funding meeting. Which do you hand
her — the citywide (pooled) rates, the within-stratum rates, or the adjusted rates — and
what is the one-sentence reason, in terms she would use?

*(1-2 sentences)*

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Part 2 (autograded): the analysts' three diagrams — backdoor paths by hand

**Why.** The department's second question: *which routes in our own diagrams let the
difference between enrollees and everyone else pass for a program effect — and are they
open?* Part 1 adjusted for neighborhood need because the department *told* you how
outreach was targeted. The analysts suspect need is not the whole story, but could not
agree on one picture, so they sent three candidate diagrams. The routes that let
confounding through are the **backdoor paths**; whether each one is open with nothing
adjusted for decides whether the naive comparison is already wrong.

| node | in the department's records |
|---|---|
| `X` | enrolled in the CHW program — the treatment |
| `Y` | any ER visit in the program year — the outcome |
| `Need` | lives in a high-need neighborhood (the Part 1 strata) |
| `Clinic` | lives near a community clinic (outreach was run out of the clinics) |
| `Primary` | kept their primary-care appointments during the program year |
| `Visits` | CHW home visits completed, logged over the year (an ER stay interrupts the schedule) |
| `Retained` | still enrolled at year end |
| `Trust` | trust in the health system, from a survey taken before the program |
| `Chronic` | has a chronic condition |
| `Screened` | attended the department's free health screening the spring *before* the program |

Every variable in a diagram is **observed** — the department could adjust for any of them.

**The path rules.** Every interior node of a path (arrow direction ignored, no node
visited twice) is one of three things **on that path**:

| structure | middle node | open by default? | conditioning on the middle node |
|---|---|---|---|
| chain $A \to B \to C$ | mediator | open | **blocks** the path |
| fork $A \leftarrow B \to C$ | confounder | open | **blocks** the path |
| collider $A \to B \leftarrow C$ | common effect | **blocked** | **opens** the path (so does conditioning on any descendant of $B$) |

A path is blocked if at least one interior node is blocked. The lecture's procedure sorts
every path from `X` to `Y` into **causal** (every arrow points away from `X` — the
program's effect; never block it), **backdoor** (first arrow points *into* `X`), or
**neither** (first arrow out of `X` but not causal — it runs into a collider).

In [ ]:
# The analysts' three candidate diagrams: (cause, effect) edges plus drawing positions.
DAGS = {
    "primary_care": dict(edges=[("Need", "X"), ("Need", "Y"), ("Clinic", "X"), ("Clinic", "Primary"),
                                ("X", "Primary"), ("Primary", "Y"), ("X", "Y")],
                         pos={"Need": (1, 1.1), "X": (0, 0), "Y": (2, 0), "Primary": (1, -0.7),
                              "Clinic": (-0.3, -1.5)}),
    "home_visits":  dict(edges=[("Need", "X"), ("Need", "Y"), ("X", "Visits"), ("Y", "Visits"),
                                ("Visits", "Retained"), ("X", "Y")],
                         pos={"Need": (1, 1.1), "X": (0, 0), "Y": (2, 0), "Visits": (1, -0.8),
                              "Retained": (1, -1.9)}),
    "screening":    dict(edges=[("Trust", "X"), ("Trust", "Screened"), ("Chronic", "Screened"),
                                ("Chronic", "Y"), ("X", "Y")],
                         pos={"Trust": (0, 1.3), "Chronic": (2, 1.3), "Screened": (1, 0.65),
                              "X": (0, 0), "Y": (2, 0)}),
}
DAG_NAMES = list(DAGS)

def dag(name):
    """The networkx DiGraph for one of the diagrams."""
    return nx.DiGraph(DAGS[name]["edges"])

def draw_dag(name, ax=None, S=()):
    """Draw a diagram; nodes in S (an adjustment set) are shaded orange."""
    G = dag(name)
    if ax is None:
        _, ax = plt.subplots(figsize=(4.5, 3.6))
    colors = ["#fdae61" if n in set(S) else "#cde" for n in G.nodes]
    nx.draw(G, DAGS[name]["pos"], ax=ax, with_labels=True, node_color=colors,
            node_size=2400, font_size=9, arrowsize=18, edgecolors="#333", linewidths=1.1)
    ax.set_title(name + (f"   S = {sorted(S)}" if S else ""), fontsize=11)
    ax.margins(0.2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, name in zip(axes, DAG_NAMES):
    draw_dag(name, ax=ax)
plt.tight_layout(); plt.show()

for name in DAG_NAMES:
    print(f"{name:13s} edges: " + ", ".join(f"{a}->{b}" for a, b in DAGS[name]["edges"]))

**Task.** For **each** of the three diagrams, list **every backdoor path** from `X` to
`Y` and mark it `"open"` or `"blocked"` **with nothing conditioned on** (forks and chains
open, colliders closed). Do **not** list causal or *neither* paths.

Write a path as a tuple of node names from `X` to `Y`: $X \leftarrow A \to B \to Y$ is
`("X", "A", "B", "Y")`. Fill in `my_backdoor_paths`, a dict mapping each name in
`DAG_NAMES` to a list of `(path, status)` pairs. Do this by reading the drawings, not by
writing a search. Order does not matter.

In [ ]:
# Type them in by hand from the drawings, one diagram at a time. Format:
#   my_backdoor_paths = {"primary_care": [(("X", "A", "B", "Y"), "blocked"), ...], ...}
...

for name in DAG_NAMES:
    print(f"{name:13s}", my_backdoor_paths.get(name))

In [ ]:
grader.check("q4")

## Part 3 (autograded): one valid and one *invalid* adjustment set per diagram

**Why.** This is the question the department will act on: *which variables in our
records should we adjust for, and which must we leave alone?* The **backdoor criterion**
answers both at once: a set $S$ is a valid adjustment set if it **blocks every backdoor
path** and contains **no descendant of `X`**. You also name one *invalid* set per
diagram, because the budget office's instinct will be "control for everything in the case
file", and your memo has to say exactly why that fails.

The next cell is **provided** — run it, don't edit it. It turns the path rules above into
code: `backdoor_paths(G)` lists every path from `X` to `Y` whose first edge points into
`X`, and `is_open(G, path, S)` says whether a path is open when you condition on the set
`S`. Its printout should agree with your Part 2 list.

In [ ]:
# Provided: the path rules as code. Run as-is.
def backdoor_paths(G):
    """All backdoor paths from X to Y in G, as tuples of node names."""
    U = G.to_undirected()
    return [tuple(p) for p in nx.all_simple_paths(U, "X", "Y") if G.has_edge(p[1], "X")]


def is_open(G, path, S=()):
    """True if `path` (tuple of nodes) is open in DAG G when conditioning on the set S."""
    S = set(S)
    for i in range(1, len(path) - 1):
        prev, node, nxt = path[i - 1], path[i], path[i + 1]
        if G.has_edge(prev, node) and G.has_edge(nxt, node):     # collider on this path
            if not (S & ({node} | nx.descendants(G, node))):
                return False
        elif node in S:                                         # chain or fork node
            return False
    return True


for name in DAG_NAMES:
    G = dag(name)
    for p in backdoor_paths(G):
        print(f"{name:13s} {'-'.join(p):28s} {'open' if is_open(G, p) else 'blocked'}")

**Task.** For each diagram propose:

- `valid_set[name]` — a set of nodes that **satisfies the backdoor criterion** for the
  total effect of `X` on `Y`. Prefer the smallest such set. It may be **empty** —
  `set()` — if nothing needs blocking.
- `invalid_set[name]` — a set that **fails** the criterion *because of something it
  contains*: a mediator, a collider, or a descendant of a collider (the "throw everything
  in" mistake).

Then complete `satisfies_backdoor(G, S)` — the criterion's two clauses, using
`is_open` and `backdoor_paths` from the provided cell — and check both sets for every
diagram: the printout should read `True` for each valid set and `False` for each invalid
one. (`satisfies_backdoor` is not autograded on its own; the sets are, against the
criterion.)

In [ ]:
def satisfies_backdoor(G, S):
    """True if S satisfies the backdoor criterion for X -> Y in G."""
    S = set(S)
    if S & (nx.descendants(G, "X") | {"X", "Y"}):
        return False
    return all(not is_open(G, p, S) for p in backdoor_paths(G))



# Fill these in by hand, e.g. valid_set = {"primary_care": {...}, ...}
valid_set = ...
invalid_set = ...

for name in DAG_NAMES:
    G = dag(name)
    print(f"{name:13s} valid {str(sorted(valid_set[name])):22s} -> {satisfies_backdoor(G, valid_set[name])}"
          f"    invalid {str(sorted(invalid_set[name])):22s} -> {satisfies_backdoor(G, invalid_set[name])}")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for j, name in enumerate(DAG_NAMES):
    draw_dag(name, ax=axes[0, j], S=valid_set[name])
    draw_dag(name, ax=axes[1, j], S=invalid_set[name])
plt.tight_layout(); plt.show()

**Discuss (participation):** for each of your three invalid sets, tell the department's
analysts in a few words *which structure* makes it a bad control — a mediator, a
collider, or a collider's descendant. Then two follow-ups they will raise:

- `primary_care`: "`Primary` blocks the clinic backdoor just as well as `Clinic` does, and
  we already have it. Why not use it?"
- `screening`: "`Screened` is associated with both enrollment and ER use, and it was
  recorded *before* the program started. Why are you telling us to adjust for nothing?"

*(a few words per diagram, one sentence per follow-up)*

In [ ]:
grader.check("q5")

<!-- BEGIN QUESTION -->

## Report out: what you would tell the commissioner

The commissioner will carry one number into the funding meeting, and the budget office
has already offered to "fix" the comparison its own way. Discuss as a team (~5 min),
then write in the cell below.

**Your memo.** Write the two sentences you would put at the top of your memo to the
commissioner: the **recommendation** (does the program appear to reduce ER visits, and
which number says so), and the **one caveat** she must hear.

Then, briefly:

1. **What to measure next.** Your Part 1 number adjusts for neighborhood need only. Under
   which of the three diagrams would it still be biased, and what would you ask the
   department to measure before you call it the program's effect?
2. **What not to adjust for.** The budget office proposes to rerun the comparison
   "controlling for everything in the case file" — need, primary-care appointments kept,
   home visits completed, still enrolled at year end, and spring screening — and promises
   a tight confidence interval. Which of those do you tell them to drop, which structure
   makes each one a bad control, and why can a tight interval not warn anyone?
3. **What she must believe.** For the adjusted number to be the program's effect, what
   has to be true about how residents came to be enrolled? Can the department's data
   check it, and what could the department do next year so it would not have to?

**Pair-share (~3 min).** Turn to a neighboring team and compare memos: same
recommendation? same caveat? the same variables dropped? Be ready to share one takeaway
with the class.

> Staff will visit tables during this part. Be ready to explain your reasoning and to
> point at the cell that supports each claim.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Wrap-up

Before you leave, each group should be able to answer, without notes:

1. In the program table, which two numbers were the confounder's fingerprints, and what
   did the adjusted rate do to them?
2. The two clauses of the backdoor criterion — and why $\{Need, Primary\}$ fails in
   `primary_care` even though it blocks every backdoor path.
3. In `screening`, why is adjusting for `Screened` worse than adjusting for nothing?

Submit the autograded parts (1–3) to Gradescope.